In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from fuzzywuzzy import fuzz

In [ ]:
AAPL = pd.read_csv("AAPL_combined.csv")
#AMZN = pd.read_csv("AMZN_combined.csv")
#FB = pd.read_csv("FB_combined.csv")
#NVDA = pd.read_csv("NVDA_combined.csv")
#TSLA = pd.read_csv("TSLA_combined.csv")

#AAPL.head(20)
#AMZN.head(20)
#FB.head(20)
#NVDA.head(20)
#TSLA.head(20)

In [ ]:
import ast

def clean_entities_sentiment(val):
    if pd.isna(val):
        return 0
    try:
        # Parse the string as a dictionary
        d = ast.literal_eval(val)
        
        # Check if 'sentiment' key exists
        if 'sentiment' in d:
            sentiment_data = d['sentiment']
            
            # If sentiment is None, return 0
            if sentiment_data is None:
                return 0
            
            # If sentiment is a dictionary, look for 'basic' key
            if isinstance(sentiment_data, dict):
                basic_sentiment = sentiment_data.get('basic')
                if basic_sentiment == 'Bullish':
                    return 1
                elif basic_sentiment == 'Bearish':
                    return -1
        
        # If we reach here, it's Neutral/None or unknown structure
        return 0
        
    except (ValueError, SyntaxError) as e:
        # Handle cases where the string isn't valid python literal
        return 0

# Apply to all dataframes
#datasets = [AAPL, AMZN, FB, NVDA, TSLA]
datasets = [APPL]
#dataset_names = ["AAPL", "AMZN", "FB", "NVDA", "TSLA"]
dataset_names = ["APPL"]
for df, name in zip(datasets, dataset_names):
    print(f"Processing {name}...")
    df['entities'] = df['entities'].apply(clean_entities_sentiment)
    print(f"Unique values in {name}['entities'] after cleaning: {df['entities'].unique()}")
    print("-" * 20)

# Verify the result on AMZN
print(AMZN[['entities']].head())

In [ ]:
#AAPL.head(20)
#AMZN.head(20)
#FB.head(20)
#NVDA.head(20)
#TSLA.head(20)

In [ ]:
# Text cleaning utilities: lowercase, remove URLs/HTML/emoji/handles/hashtags, normalize unicode, expand contractions
import re
import html
import unicodedata
import pandas as pd

# small contractions map (add more as needed)
_CONTRACTIONS = {
    "can't": "cannot",
    "won't": "will not",
    "i'm": "i am",
    "it's": "it is",
    "that's": "that is",
    "you're": "you are",
    "we're": "we are",
    "they're": "they are",
    "she's": "she is",
    "he's": "he is",
    "ain't": "is not",
    "let's": "let us",
    "'ve": " have",
    "'re": " are",
    "'ll": " will",
    "'d": " would",
    "n't": " not"
}

_CONTRACTIONS_RE = re.compile('|'.join(re.escape(k) for k in _CONTRACTIONS.keys()), flags=re.IGNORECASE)

def _expand_contractions(text: str) -> str:
    def _repl(m):
        key = m.group(0).lower()
        return _CONTRACTIONS.get(key, key)
    return _CONTRACTIONS_RE.sub(_repl, text)

# basic emoji range pattern (covers common emoji ranges)
_EMOJI_RE = re.compile("[\U0001F300-\U0001F6FF\U0001F900-\U0001F9FF\U0001F1E0-\U0001F1FF]+", flags=re.UNICODE)

def clean_text(text, *, remove_urls=True, remove_html=True, remove_emoji=True, remove_handles=True, remove_hashtags=True, expand_contractions=True) -> str:
    if pd.isna(text):
        return ""
    s = str(text)
    # normalize unicode and unescape HTML entities
    s = unicodedata.normalize('NFKC', s)
    s = html.unescape(s)
    # lower for consistent processing
    s = s.lower()
    # expand contractions (before removing punctuation)
    if expand_contractions:
        s = _expand_contractions(s)
    # remove urls
    if remove_urls:
        s = re.sub(r'https?://\S+|www\.\S+', '', s)
    # remove html tags
    if remove_html:
        s = re.sub(r'<[^>]+>', '', s)
    # handle user handles
    if remove_handles:
        s = re.sub(r'@\w+', '', s)
    # hashtags: either remove entirely or keep text without '#'.
    if remove_hashtags:
        s = re.sub(r'#\w+', '', s)
    else:
        s = re.sub(r'#', '', s)
    # remove emoji
    if remove_emoji:
        s = _EMOJI_RE.sub('', s)
    # remove leftover non-printable/control characters
    s = re.sub(r'[\r\t\x0b\x0c]', ' ', s)
    # collapse whitespace
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def clean_series(series: pd.Series, **kwargs) -> pd.Series:
    return series.fillna('').astype(str).map(lambda x: clean_text(x, **kwargs))

# Example usage (uncomment to run):
APPL['clean_text'] = clean_series(AMZN['body'])
print(APPL['clean_text'].head())

In [ ]:
# Clean the 'body' column (social media posts) for all datasets

print("Cleaning 'body' column for all datasets...")
for df, name in zip(datasets, dataset_names):
    print(f"\nProcessing {name}...")
    df['body_clean'] = clean_series(df['body'])
    print(f"Sample cleaned text from {name}:")
    print(f"  Original: {df['body'].iloc[0][:100]}...")
    print(f"  Cleaned:  {df['body_clean'].iloc[0][:100]}...")

print("\nCleaning complete!")


A body clean column is added at the end


In [ ]:
#AAPL.head(20)
#AMZN.head(20)
#FB.head(20)
#NVDA.head(20)
#TSLA.head(20)

Remove


Check for duplicate values


In [ ]:
# Duplicate Detection
from collections import Counter

print("=" * 60)
print("DUPLICATE DETECTION")
print("=" * 60)

for df, name in zip(datasets, dataset_names):
    print(f"\n{name}:")
    print(f"  Total rows: {len(df)}")
    
    # Exact duplicates based on 'body_clean'
    exact_dups = df.duplicated(subset=['body_clean'], keep=False).sum()
    print(f"  Exact duplicate rows (body_clean): {exact_dups}")
    
    # Exact duplicates based on 'body' (original)
    exact_dups_orig = df.duplicated(subset=['body'], keep=False).sum()
    print(f"  Exact duplicate rows (original body): {exact_dups_orig}")
    
    # Check for duplicate user-content pairs (same user posting same content)
    user_content_dups = df.duplicated(subset=['user', 'body_clean'], keep=False).sum()
    print(f"  User-content duplicate rows: {user_content_dups}")
    
    # Remove exact duplicates (keep first occurrence)
    df_deduped = df.drop_duplicates(subset=['body_clean'], keep='first')
    duplicates_removed = len(df) - len(df_deduped)
    print(f"  Rows after removing exact duplicates: {len(df_deduped)} - removed {duplicates_removed}")
    
    # Show most repeated content
    if exact_dups > 0:
        top_repeats = df[df.duplicated(subset=['body_clean'], keep=False)]['body_clean'].value_counts().head(3)
        print(f"  Most repeated posts:")
        for idx, (text, count) in enumerate(top_repeats.items(), 1):
            print(f"    {idx}. Posted {count} times: {text[:75]}...")

# Optional: Apply deduplication to all datasets
print("\n" + "=" * 60)
apply_dedup = True  # Set to True to remove duplicates
if apply_dedup:
    print("Removing exact duplicates from all datasets...")
    for i, (df, name) in enumerate(zip(datasets, dataset_names)):
        original_len = len(df)
        datasets[i] = df.drop_duplicates(subset=['body_clean'], keep='first')
        print(f"  {name}: {original_len} -> {len(datasets[i])} rows")
    print("Deduplication complete!")
else:
    print("Deduplication NOT applied. Set apply_dedup=True to remove duplicates.")

Remove Stop Words


In [ ]:
# Remove Stop Words
%pip install nltk

from nltk.corpus import stopwords
import nltk

# Download stopwords (run once)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

# Get English stop words
stop_words = set(stopwords.words('english'))

# Add custom financial stop words that may not be useful
custom_stop_words = {'stock', 'price', 'market', 'will', 'would', 'could', 'said', 'say'}
stop_words.update(custom_stop_words)

def remove_stopwords(text):
    """Remove stop words from text"""
    if pd.isna(text) or text == "":
        return ""
    words = text.split()
    filtered_words = [word for word in words if word.lower() not in stop_words]
    return ' '.join(filtered_words)

print("=" * 60)
print("REMOVING STOP WORDS")
print("=" * 60)

for df, name in zip(datasets, dataset_names):
    print(f"\n{name}:")
    df['body_no_stopwords'] = df['body_clean'].apply(remove_stopwords)
    
    # Show comparison
    sample_idx = 0
    print(f"  Sample before: {df['body_clean'].iloc[sample_idx][:100]}")
    print(f"  Sample after:  {df['body_no_stopwords'].iloc[sample_idx][:100]}")
    
    # Count average words removed
    before_count = df['body_clean'].apply(lambda x: len(str(x).split())).mean()
    after_count = df['body_no_stopwords'].apply(lambda x: len(str(x).split())).mean()
    print(f"  Avg words before: {before_count:.1f} → after: {after_count:.1f} ({100*(before_count-after_count)/before_count:.1f}% removed)")

print("\nStop word removal complete!")

In [ ]:
#AAPL.head(20)
#AMZN.head(20)
#FB.head(20)
#NVDA.head(20)
#TSLA.head(20)

Termoral features


In [ ]:
# Temporal Features Engineering
print("=" * 60)
print("TEMPORAL FEATURES EXTRACTION")
print("=" * 60)

# First, identify the timestamp column
timestamp_cols = ['created_at', 'timestamp', 'date', 'published_at', 'post_date']
time_col = None

for df, name in zip(datasets, dataset_names):
    print(f"\n{name} columns: {df.columns.tolist()}")
    # Find which column is the timestamp
    for col in timestamp_cols:
        if col in df.columns:
            time_col = col
            print(f"  Found timestamp column: '{time_col}'")
            break
    if time_col:
        break

if time_col:
    print(f"\nUsing '{time_col}' as timestamp column for all datasets")
    
    for df, name in zip(datasets, dataset_names):
        print(f"\nProcessing {name}...")
        
        # Convert to datetime
        df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
        
        # ===== TIME-OF-DAY FEATURES =====
        df['hour'] = df[time_col].dt.hour
        df['minute'] = df[time_col].dt.minute
        df['time_period'] = pd.cut(df['hour'], 
                                    bins=[0, 6, 12, 18, 24], 
                                    labels=['night', 'morning', 'afternoon', 'evening'],
                                    right=False)
        
        # ===== DATE FEATURES =====
        df['day_of_week'] = df[time_col].dt.dayofweek  # 0=Monday, 6=Sunday
        df['day_name'] = df[time_col].dt.day_name()
        df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
        df['day_of_month'] = df[time_col].dt.day
        df['month'] = df[time_col].dt.month
        df['quarter'] = df[time_col].dt.quarter
        df['week_of_year'] = df[time_col].dt.isocalendar().week
        df['year'] = df[time_col].dt.year
        
        # ===== TREND FEATURES =====
        # Posts per hour (rolling window)
        df['posts_per_hour'] = df.groupby(df[time_col].dt.floor('h')).cumcount() + 1
        
        # Sentiment trend (rolling average using groupby)
        if 'entities' in df.columns:
            # Sort by timestamp first for rolling calculation
            df = df.sort_values(by=time_col).reset_index(drop=True)
            # Calculate rolling mean with a fixed window of 24 rows instead of 24 hours
            df['sentiment_rolling_mean_24'] = df['entities'].rolling(window=24, min_periods=1).mean()
        
        # Time since last post (in minutes)
        df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
        df['time_since_last_post'] = df[time_col].diff().dt.total_seconds() / 60
        df['time_since_last_post'] = df['time_since_last_post'].fillna(0)
        
        # Cumulative posts by user over time
        df['user_post_count'] = df.groupby('user').cumcount() + 1
        
        print(f"  ✓ Temporal features created successfully")
        print(f"  Sample time features:")
        print(f"    Hour range: {df['hour'].min():.0f} to {df['hour'].max():.0f}")
        print(f"    Day of week: {sorted(df['day_of_week'].unique())}")
        print(f"    Weekend posts: {df['is_weekend'].sum()} / {len(df)}")
        print(f"    Avg posts per hour: {df['posts_per_hour'].mean():.2f}")
        print(f"    Max user post count: {df['user_post_count'].max():.0f}")
else:
    print("ERROR: Could not find timestamp column in data!")
    print("Expected one of:", timestamp_cols)


In [ ]:
# Display temporal features
print("=" * 60)
print("TEMPORAL FEATURES SUMMARY")
print("=" * 60)

for df, name in zip(datasets, dataset_names):
    print(f"\n{name} - Sample of temporal features:")
    temporal_cols = ['hour', 'day_of_week', 'day_name', 'is_weekend', 'month', 
                     'posts_per_hour', 'time_since_last_post', 'user_post_count']
    
    # Show columns that exist
    existing_cols = [col for col in temporal_cols if col in df.columns]
    if existing_cols:
        print(df[existing_cols].head(10))
    
    if 'hour' in df.columns:
        print(f"\n{name} - Temporal features stats:")
        print(f"  Hour distribution:\n{df['hour'].value_counts().sort_index()}")
        if 'day_name' in df.columns:
            print(f"  Posts by day of week:\n{df['day_name'].value_counts()}")


In [ ]:
#AAPL.to_csv("AAPL_final.csv", index=False)
#AMZN.to_csv("AMZN_final.csv", index=False)
#FB.to_csv("FB_final.csv", index=False)
#NVDA.to_csv("NVDA_final.csv", index=False)
#TSLA.to_csv("TSLA_final.csv", index=False)

In [ ]:
import ast, re, html, unicodedata
import pandas as pd
from nltk.corpus import stopwords

# ── Sentiment encoder ────────────────────────────────────────
def parse_sentiment(val):
    if pd.isna(val):
        return 0
    s = str(val).strip()
    if s in ('1', '-1', '0'):
        return int(s)
    try:
        d = ast.literal_eval(s)
        basic = (d.get('sentiment') or {}).get('basic') if isinstance(d, dict) else None
        if basic == 'Bullish':  return  1
        if basic == 'Bearish':  return -1
    except Exception:
        pass
    return 0

# ── Text cleaner ─────────────────────────────────────────────
_EMOJI_RE = re.compile(r'[\U0001F300-\U0001F6FF\U0001F900-\U0001F9FF\U0001F1E0-\U0001F1FF]+')

def clean_text(text):
    if pd.isna(text):
        return ''
    s = unicodedata.normalize('NFKC', str(text))
    s = html.unescape(s).lower()
    s = re.sub(r'https?://\S+|www\.\S+', '', s)
    s = re.sub(r'<[^>]+>', '', s)
    s = re.sub(r'@\w+', '', s)
    s = re.sub(r'#\w+', '', s)
    s = _EMOJI_RE.sub('', s)
    return re.sub(r'\s+', ' ', s).strip()

# ── Stop words ───────────────────────────────────────────────
stop_words = set(stopwords.words('english')) | {'stock', 'price', 'market', 'will', 'would', 'could', 'said', 'say'}

def remove_stopwords(text):
    return ' '.join(w for w in str(text).split() if w not in stop_words)

# ── Main pipeline ────────────────────────────────────────────
#stocks = ['AAPL', 'AMZN', 'FB', 'NVDA', 'TSLA']
stocks = ['APPL']
for name in stocks:
    print(f"Processing {name}...")
    df = pd.read_csv(f"{name}_combined.csv")

    before = len(df)

    # 1. Clean entities
    df['entities'] = df['entities'].apply(parse_sentiment)

    # 2. Clean text
    df['body_clean'] = df['body'].apply(clean_text)

    # 3. Remove stop words
    df['body_no_stopwords'] = df['body_clean'].apply(remove_stopwords)

    # 4. Drop duplicates (keep first occurrence of each unique cleaned post)
    df = df.drop_duplicates(subset=['body_clean'], keep='first')

    after = len(df)

    df.to_csv(f"{name}_cleaned.csv", index=False)
    print(f"  {before:>10,} rows → {after:>10,} rows  (removed {before - after:,})  saved to {name}_cleaned.csv")

print("\nDone.")

In [ ]:
# Bullish / Bearish count per stock
import pandas as pd
import matplotlib.pyplot as plt

sentiment_summary = {}
amazon = "APPL_cleaned.csv"
dataset_names = ["APPL"] 
for df, name in zip([pd.read_csv(amazon)], dataset_names):
    counts = df['entities'].value_counts()
    sentiment_summary[name] = {
        'Bullish':  counts.get(1,  0),
        'Bearish':  counts.get(-1, 0),
        'Neutral':  counts.get(0,  0),
        'Total':    len(df),
    }

summary_df = pd.DataFrame(sentiment_summary).T
summary_df['Bullish %'] = (summary_df['Bullish'] / summary_df['Total'] * 100).round(2)
summary_df['Bearish %'] = (summary_df['Bearish'] / summary_df['Total'] * 100).round(2)
summary_df['Neutral %'] = (summary_df['Neutral'] / summary_df['Total'] * 100).round(2)

print("=" * 65)
print("BULLISH / BEARISH COUNT PER STOCK")
print("=" * 65)
print(summary_df[['Bullish', 'Bearish', 'Neutral', 'Total',
                   'Bullish %', 'Bearish %', 'Neutral %']].to_string())

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute counts
summary_df[['Bullish', 'Bearish', 'Neutral']].plot(
    kind='bar', ax=axes[0],
    color=['#2ecc71', '#e74c3c', '#95a5a6'], edgecolor='black'
)
axes[0].set_title('Sentiment Count per Stock (absolute)')
axes[0].set_xlabel('Stock')
axes[0].set_ylabel('Number of posts')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend()

# Percentage
summary_df[['Bullish %', 'Bearish %', 'Neutral %']].plot(
    kind='bar', ax=axes[1],
    color=['#2ecc71', '#e74c3c', '#95a5a6'], edgecolor='black'
)
axes[1].set_title('Sentiment Distribution per Stock (%)')
axes[1].set_xlabel('Stock')
axes[1].set_ylabel('% of posts')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

#stocks = ['AAPL', 'AMZN', 'FB', 'NVDA', 'TSLA']
stock = ["APPL"]
print(f"{'Stock':<6} {'Combined':>10} {'Final':>10} {'Rows removed':>13}  {'Deduped?':<16} {'Entities clean?'}")
print("-" * 75)

for name in stocks:
    combined = pd.read_csv(f"{name}_combined.csv", usecols=['entities', 'body'])
    final    = pd.read_csv(f"{name}_cleaned.csv",    usecols=['entities', 'body'])

    diff    = len(combined) - len(final)
    deduped = "YES" if diff > 0 else "NO - same size"

    # entities in final should only contain -1, 0, 1
    ent_vals = set(final['entities'].dropna().unique())
    expected = {-1, 0, 1}
    extra    = ent_vals - expected
    entities_ok = "YES" if not extra else f"NO - unexpected: {extra}"

    print(f"{name:<6} {len(combined):>10,} {len(final):>10,} {diff:>13,}  {deduped:<16} {entities_ok}")

In [ ]:
# this should how many duplicates existed in the cleaned

dataset = "APPL_cleaned.csv"
df = pd.read_csv(dataset)

duplicates = df.duplicated(subset=['body_clean'], keep=False).sum()
print(f"Total duplicate rows in {dataset}: {duplicates}")

